# DoQ Tunneling Detection — End-to-End Pipeline

One sectioned Colab notebook: **setup → generate → extract/merge → train → export**.

Data collection plan and citation bundle: `EXECUTION_PLAN.md` and `METHODOLOGY_CITATIONS.md` (project root).

## 0. Setup

Installs system tools, Python deps, Go, downloads `dnsproxy`/`dnspyre`, builds `dnstt-fork`, mounts Google Drive, and clones the pipeline repo.

In [ ]:
import os, json, shutil, subprocess, urllib.request
from google.colab import drive

DS = '/content/drive/MyDrive/doq-dataset'
if not os.path.isdir('/content/drive/MyDrive'):
    drive.mount('/content/drive')
os.makedirs(DS, exist_ok=True)

# system tools
!apt-get update -qq > /dev/null && apt-get install -y -qq tcpdump tshark git build-essential > /dev/null

# python deps
!python3 -m pip install -q dnspython aioquic scapy pandas scikit-learn xgboost imbalanced-learn torch

# Go >= 1.24 for dnstt-fork (skip if already installed)
!test -d /usr/local/go || (wget -q https://go.dev/dl/go1.24.0.linux-amd64.tar.gz -O /tmp/go.tar.gz && tar -C /usr/local -xzf /tmp/go.tar.gz)
os.environ['PATH'] += ':/usr/local/go/bin'

# pipeline repo (skip clone if already present)
REPO_URL = "https://github.com/Cyberguru1/doq-tunnel-dataset.git"
REPO = '/content/doq-tunnel-dataset'
if not os.path.isdir(REPO):
    !git clone $REPO_URL $REPO 2>/dev/null || true
os.chdir(REPO)
os.makedirs('bin', exist_ok=True)

# latest prebuilt binaries via GitHub API
def latest_asset(repo, contains):
    try:
        with urllib.request.urlopen(f"https://api.github.com/repos/{repo}/releases/latest") as r:
            data = json.load(r)
        for a in data.get("assets", []):
            if contains in a["name"]:
                return a["browser_download_url"]
    except Exception:
        pass
    return None

for repo, contains, exe in [("AdguardTeam/dnsproxy", "linux-amd64", "dnsproxy"),
                            ("Tantalor93/dnspyre", "linux-amd64", "dnspyre")]:
    url = latest_asset(repo, contains)
    if not url:
        print("skipped", exe, "(no matching asset)")
        continue
    subprocess.run(f"wget -q '{url}' -O /tmp/{exe}.tar.gz && tar -C bin -xzf /tmp/{exe}.tar.gz", shell=True)
    # tarballs may extract into a subdir (e.g. linux-amd64/dnsproxy); flatten to bin/
    for root, _dirs, files in os.walk('bin'):
        if exe in files and root != 'bin':
            shutil.copy(os.path.join(root, exe), os.path.join('bin', exe))
            break
    print("installed", exe)

# build dnstt-fork (fresh clone each run, output to separate dir)
!mkdir -p /tmp/dnstt-bin
!rm -rf /tmp/dnstt && git clone -q https://github.com/IvanTopGaming/dnstt /tmp/dnstt && cd /tmp/dnstt && go build -o /tmp/dnstt-bin/dnstt-server ./dnstt-server && go build -o /tmp/dnstt-bin/dnstt-client ./dnstt-client
!cp -f /tmp/dnstt-bin/dnstt-server /tmp/dnstt-bin/dnstt-client bin/
print("bin:", sorted(os.listdir('bin')))


In [ ]:
# ===== 0.5 Download benign DoQ (website-fingerprinting-doq traces) =====
# Real live DoQ (RFC 9250, UDP 853, ALPN "doq") captured to AdGuard DNS.
# Login-free alternative to the IEEE DataPort dataset.
import os, subprocess

DST = f'{DS}/doq_alt'
os.makedirs(DST, exist_ok=True)
BASE = 'https://raw.githubusercontent.com/svsdlmao/website-fingerprinting-doq/main/data/raw'

for name in ['doq_traces.jsonl', 'doq_traces_background.jsonl', 'doq_traces_browser.jsonl']:
    dst = os.path.join(DST, name)
    if not os.path.exists(dst):
        subprocess.run(['wget', '-q', f'{BASE}/{name}', '-O', dst])
        print('downloaded', name)
    else:
        print('already have', name)

print('jsonl files:', sorted(f for f in os.listdir(DST) if f.endswith('.jsonl')))


## 1. Generate malicious DoQ (dnstt-fork `-doq`)

Runs the loopback tunnel: curl → dnstt-client → DoQ → dnsproxy resolver → dnstt-server → TCP sink. Fully offline, no public domain. ~2 min per config; collect at least 3 config labels.

In [ ]:
import os, subprocess

BIN = os.path.abspath('bin')
os.chdir('/content/doq-tunnel-dataset')

for label in ['fast', 'normal', 'slow']:
    out = f'{DS}/malicious_doq_{label}.pcap'
    print("generating", label, "->", out)
    r = subprocess.run(['bash', 'src/generate_malicious.sh', out, label, '120'],
                       env={**os.environ, 'BIN_DIR': BIN})
    print("done", label, "rc=", r.returncode)

## 2. Generate benign DoQ (lab)

Benign DoQ via a local dnsproxy resolver + dnspython[doq] client. This lab benign is REQUIRED alongside the IEEE real benign so lab-vs-real is not a shortcut for the classifier.

In [ ]:
import os, subprocess

BIN = os.path.abspath('bin')
out = f'{DS}/benign_doq_lab.pcap'
print("generating benign ->", out)
r = subprocess.run(['bash', 'src/generate_benign.sh', out, '120'],
                   env={**os.environ, 'BIN_DIR': BIN})
print("done, rc=", r.returncode)

## 3. Extract features & merge (with IEEE benign DoQ)

Runs every source through the SAME 28-feature extractor. Edit `IEEE_PATH` to your unzipped DoQ+QUIC packet CSVs (IEEE DataPort DOI 10.21227/km5h-g294).

In [ ]:
import os, json, glob

os.chdir('/content/doq-tunnel-dataset')
DOQ_ALT_PATH = f'{DS}/doq_alt/*.jsonl'   # benign DoQ traces (login-free)

sources = {
    "window_sec": 10.0,
    "idle_timeout": 120.0,
    "min_packets": 2,
    "sources": [
        {"kind": "pcap", "paths": sorted(glob.glob(f'{DS}/malicious_doq_*.pcap')),
         "class": "malicious_doq", "tool": "dnstt", "source": "lab_dnstt"},
        {"kind": "pcap", "paths": sorted(glob.glob(f'{DS}/benign_doq_lab*.pcap')),
         "class": "benign_doq", "tool": None, "source": "lab_dnsproxy"},
        {"kind": "doq_jsonl", "paths": sorted(glob.glob(DOQ_ALT_PATH)),
         "class": "benign_doq", "tool": None, "source": "doq_traces"},
    ],
}

with open('/tmp/sources.json', 'w') as f:
    json.dump(sources, f, indent=2)

!python3 src/merge_dataset.py --config /tmp/sources.json --out {DS}/combined.csv

## 4. Train & evaluate

Baselines first (RF / XGBoost / MLP) on the 28 flow features with 5-fold stratified CV. The BiLSTM/dilated-conv sequence model is milestone M5 (it needs per-packet sequences, not the tabular window features).

In [ ]:
import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.neural_network import MLPClassifier
from sklearn.model_selection import StratifiedKFold, cross_val_score
from xgboost import XGBClassifier

df = pd.read_csv(f'{DS}/combined.csv')
label_cols = ['class', 'tool', 'source', 'flow_id', 'window_start_ts']
feats = [c for c in df.columns if c not in label_cols]
X, y = df[feats], df['class']
print("classes:")
print(y.value_counts().to_string())

models = {
    'RandomForest': RandomForestClassifier(n_estimators=200, n_jobs=-1, random_state=0),
    'XGBoost': XGBClassifier(n_estimators=200, eval_metric='logloss', random_state=0),
    'MLP': MLPClassifier(hidden_layer_sizes=(64, 32), max_iter=200, random_state=0),
}
cv = StratifiedKFold(5, shuffle=True, random_state=0)
for name, clf in models.items():
    try:
        scores = cross_val_score(clf, X, y, cv=cv, scoring='f1_macro', n_jobs=-1)
        print(f"{name}: F1-macro = {scores.mean():.4f} +/- {scores.std():.4f}")
    except Exception as e:
        print(f"{name}: failed - {e}")

## 5. Export

Artifacts are already on Google Drive (`DS`). Verify the DOI and licensing steps before publishing (see METHODOLOGY_CITATIONS.md).

In [ ]:
print("artifacts in", DS)
!ls -lh {DS}